# Day 28 — 분석 계획 확정: 결과를 보기 전에 정하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-28.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-28.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: 반복 측정을 독립으로 세면 생기는 일

두 군 사이에 **진짜 차이가 없는** 가상 세계를 2,000번 만들어, 두 가지 방법으로 검정합니다. 차이가 없으니 "유의하다(p < 0.05)"고 나오는 비율은 5% 근처여야 정상입니다.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
n_sim, n_person, n_rec = 2000, 10, 20     # 군당 10명, 한 사람당 기록 20개
false_pos_records = 0
false_pos_persons = 0

for _ in range(n_sim):
    # 두 군 사이에 진짜 차이는 '없다'. 사람마다 고유한 수준(person effect)만 다르다.
    def make_group():
        person_level = rng.normal(0, 1.0, n_person)                              # 사람 간 차이
        return person_level[:, None] + rng.normal(0, 0.5, (n_person, n_rec))     # 사람 내 흔들림
    a, b = make_group(), make_group()

    # 잘못된 방법: 기록 200개씩을 독립 표본처럼 비교
    if stats.ttest_ind(a.ravel(), b.ravel()).pvalue < 0.05:
        false_pos_records += 1
    # 올바른 방법: 사람별 평균 10개씩을 비교
    if stats.ttest_ind(a.mean(axis=1), b.mean(axis=1)).pvalue < 0.05:
        false_pos_persons += 1

print(f"기록 단위 비교의 거짓 양성 비율: {false_pos_records / n_sim:.1%}")
print(f"참가자 단위 비교의 거짓 양성 비율: {false_pos_persons / n_sim:.1%}")

### 예시 2: 줄 단위 split과 참가자 단위 split

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
ids = np.repeat([f"S{i:02d}" for i in range(1, 11)], 6)      # 10명 × 기록 6개 = 60줄
df = pd.DataFrame({"participant_id": ids, "window": np.tile(np.arange(6), 10)})

# 잘못된 방법: 줄(window)을 무작위로 섞어 80/20
shuffled = df.sample(frac=1, random_state=0)
train_bad, test_bad = shuffled.iloc[:48], shuffled.iloc[48:]
overlap_bad = set(train_bad["participant_id"]) & set(test_bad["participant_id"])

# 올바른 방법: '사람'을 섞어 8명/2명으로 나눈 뒤, 그 사람의 줄을 통째로 보낸다
people = rng.permutation(df["participant_id"].unique())
test_people = set(people[:2])
train_ok = df[~df["participant_id"].isin(test_people)]
test_ok = df[df["participant_id"].isin(test_people)]
overlap_ok = set(train_ok["participant_id"]) & set(test_ok["participant_id"])

print("줄 단위 split → train·test 양쪽에 다 있는 사람 수:", len(overlap_bad))
print("참가자 단위 split → 양쪽에 다 있는 사람 수:", len(overlap_ok))
print("참가자 단위 split 크기: train", len(train_ok), "줄 / test", len(test_ok), "줄")

## 🤖 AI와 함께 하기 (주간 실습, 20–30분)

3. **AI 코드를 돌리기 전에 sampling rate를 데이터에서 역산해 봅니다.** 시간 열이 있다면 간격의 역수가 sampling rate입니다. 아래는 가상 기록으로 확인하는 예입니다.

In [ ]:
import numpy as np

# 가상 기록: 100 Hz로 3초 동안 잰 시간 열(초 단위)과 힘 신호
t = np.arange(0, 3, 0.01)
force = 400 + 50*np.sin(2*np.pi*1.0*t)   # 1 Hz로 오르내리는 가상 신호

fs = 1 / np.median(np.diff(t))           # 시간 간격에서 sampling rate 역산
print(f"추정 sampling rate: {fs:.1f} Hz")
print(f"샘플 수 {len(t)}개 / fs = {len(t)/fs:.2f}초 (기록 길이와 맞는지 확인)")
print(f"시간 열의 마지막 값: {t[-1]:.2f}초")